# Phase 0: Intentional Failure 

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

print("All libraries loaded ✓")

# Load data - absolute path
train_txn = pd.read_csv(r'D:\fraud-detection-lab\data\train_transaction.csv')
train_id = pd.read_csv(r'D:\fraud-detection-lab\data\train_identity.csv')

print(f"Train transaction shape: {train_txn.shape}")
print(f"Train identity shape: {train_id.shape}")
print(f"Fraud rate: {train_txn['isFraud'].mean()*100:.2f}%")

All libraries loaded ✓
Train transaction shape: (590540, 394)
Train identity shape: (144233, 41)
Fraud rate: 3.50%


## Experiment 0A - The Naive Model

This is intentional. We're going to build a deliberately bad model and watch it lie to us.

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Step 1: Merge transaction + identity
df = train_txn.merge(train_id, on='TransactionID', how='left')

# Step 2: Minimal prep - drop non-numeric and fill nulls
X = df.drop(columns=['isFraud', 'TransactionID'])
y = df['isFraud']

X = X.select_dtypes(include=[np.number]).fillna(-999)

# Step 3: Split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# Step 4: Train XGBoost with zero special handling
model = xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
model.fit(X_train, y_train)

# Step 5: Evaluate
y_pred = model.predict(X_val)

print("=== NAIVE MODEL RESULTS ===")
print(f"Accuracy: {accuracy_score(y_val, y_pred)*100:.2f}%")
print()
print(classification_report(y_val, y_pred, target_names=['Not Fraud', 'Fraud']))

=== NAIVE MODEL RESULTS ===
Accuracy: 97.98%

              precision    recall  f1-score   support

   Not Fraud       0.98      1.00      0.99    113866
       Fraud       0.91      0.49      0.63      4242

    accuracy                           0.98    118108
   macro avg       0.94      0.74      0.81    118108
weighted avg       0.98      0.98      0.98    118108



## This is exactly the lie to be expose.

Look at what happened:

- **Accuracy:** `97.98%` ← sounds amazing  
- **Fraud Recall:** `0.49` ← model misses **51%** of actual frauds  

The model is essentially saying *"everything is fine"* half the time when fraud is actually happening.

A business would be losing money while celebrating **98% accuracy**.

---

## The math of the lie

- `4,242` fraud cases in validation  
- Model caught only `~2,078` of them  
- Missed `~2,164` real frauds  

At **₹50,000** loss per missed fraud → **₹10.8 crore** in undetected fraud

In [3]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_val, y_pred)
TN, FP, FN, TP = cm.ravel()

FP_cost = 500      # blocking a legit transaction
FN_cost = 50000    # missing a fraud

total_cost = (FP * FP_cost) + (FN * FN_cost)

print("=== CONFUSION MATRIX ===")
print(f"True Negatives  (correct legit):    {TN:,}")
print(f"False Positives (blocked legit):    {FP:,}")
print(f"False Negatives (missed fraud):     {FN:,}")
print(f"True Positives  (caught fraud):     {TP:,}")
print()
print("=== BUSINESS COST ===")
print(f"Missed frauds:       {FN:,}")
print(f"Cost per miss:       ₹{FN_cost:,}")
print(f"Total fraud loss:    ₹{FN * FN_cost:,.0f}")
print(f"False alarm cost:    ₹{FP * FP_cost:,.0f}")
print(f"Total business cost: ₹{total_cost:,.0f}")

=== CONFUSION MATRIX ===
True Negatives  (correct legit):    113,651
False Positives (blocked legit):    215
False Negatives (missed fraud):     2,176
True Positives  (caught fraud):     2,066

=== BUSINESS COST ===
Missed frauds:       2,176
Cost per miss:       ₹50,000
Total fraud loss:    ₹108,800,000
False alarm cost:    ₹107,500
Total business cost: ₹108,907,500


## Experiment 0A - Post Mortem

₹10.89 crore in losses. With a model someone called "98% accurate."
This is the exact moment in real interviews where you say: "Accuracy is a vanity metric on imbalanced data. Here's what it actually costs."

Key insight from these numbers:

- False alarm cost: ₹1.07 lakh
- Missed fraud cost: ₹10.88 crore
- Missing fraud is 1,012x more expensive than a false alarm

This means our model should be heavily biased toward catching fraud even at the cost of more false alarms. That drives every decision we make going forward.

- **Hypothesis:** XGBoost on raw features will perform well.
- **Result:** 97.98% accuracy, 49% fraud recall.
- **Business cost:** ₹10.89 crore in missed fraud on validation set alone.

**Insight:** Accuracy is meaningless on imbalanced data.
A model predicting "not fraud" on everything gets 96.5% accuracy.
The right metrics are: Recall, PR-AUC, and Expected Business Cost.

## Experiment 0B - The Leaky Model
**What we're doing:** 
Artificially creating a feature that "leaks" future information into the model. AUC will look incredible. Then we'll simulate production and watch it collapse.

In [4]:
# ============================================
# EXPERIMENT 0B — LEAKY MODEL
# ============================================

# The Leak: compute mean fraud rate per card (ProductCD)
# Problem: we're computing this on the ENTIRE dataset before splitting
# In production, future transactions don't exist yet — this is illegal

# Step 1: Create leaky feature BEFORE splitting (the mistake)
df['card_fraud_rate'] = df.groupby('card1')['isFraud'].transform('mean')

# Step 2: Also add a legitimate-looking time feature
df['hour'] = df['TransactionDT'] % 86400 // 3600  # hour of day

# Step 3: Prep features
X_leak = df.drop(columns=['isFraud', 'TransactionID'])
y_leak = df['isFraud']
X_leak = X_leak.select_dtypes(include=[np.number]).fillna(-999)

# Step 4: Split AFTER creating leaky feature (wrong order)
X_train_l, X_val_l, y_train_l, y_val_l = train_test_split(
    X_leak, y_leak, test_size=0.2, random_state=42
)

# Step 5: Train
model_leak = xgb.XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
model_leak.fit(X_train_l, y_train_l)

# Step 6: Evaluate
from sklearn.metrics import roc_auc_score

y_pred_leak = model_leak.predict(X_val_l)
y_prob_leak = model_leak.predict_proba(X_val_l)[:, 1]

print("=== LEAKY MODEL RESULTS ===")
print(f"Accuracy:  {accuracy_score(y_val_l, y_pred_leak)*100:.2f}%")
print(f"ROC-AUC:   {roc_auc_score(y_val_l, y_prob_leak):.4f}")
print()
print(classification_report(y_val_l, y_pred_leak, target_names=['Not Fraud', 'Fraud']))

=== LEAKY MODEL RESULTS ===
Accuracy:  98.12%
ROC-AUC:   0.9556

              precision    recall  f1-score   support

   Not Fraud       0.98      1.00      0.99    113866
       Fraud       0.90      0.54      0.67      4242

    accuracy                           0.98    118108
   macro avg       0.94      0.77      0.83    118108
weighted avg       0.98      0.98      0.98    118108



In [5]:
from sklearn.metrics import recall_score

# ============================================
# SIDE BY SIDE COMPARISON
# ============================================

y_prob_naive = model.predict_proba(X_val)[:, 1]

print("=== NAIVE vs LEAKY ===")
print(f"{'Metric':<20} {'Naive':>10} {'Leaky':>10}")
print("-" * 42)
print(f"{'Accuracy':<20} {'97.98%':>10} {'98.12%':>10}")
print(f"{'ROC-AUC':<20} {roc_auc_score(y_val, y_prob_naive):>10.4f} {roc_auc_score(y_val_l, y_prob_leak):>10.4f}")
print(f"{'Fraud Recall':<20} {'0.49':>10} {'0.54':>10}")
print()
print("Leaky model looks better on every metric.")
print("Now let's simulate what happens in production...")

# ============================================
# PRODUCTION SIMULATION — THE COLLAPSE
# ============================================

# In production: new transactions come in AFTER training period
# card_fraud_rate was computed on full historical data
# For NEW cards never seen before → this feature is unknown → we fill with -999
# For seen cards → we only know PAST fraud rate, not future

# Simulate: 30% of production cards are new (never seen in training)
import numpy as np

np.random.seed(42)
new_card_mask = np.random.random(len(X_val_l)) < 0.30

# Corrupt the leaky feature for new cards (simulating real production)
X_val_production = X_val_l.copy()
X_val_production.loc[X_val_production.index[new_card_mask], 'card_fraud_rate'] = -999

# Score on production data
y_prob_production = model_leak.predict_proba(X_val_production)[:, 1]
y_pred_production = model_leak.predict(X_val_production)

print("\n=== PRODUCTION PERFORMANCE (30% new cards) ===")
print(f"ROC-AUC in validation:  {roc_auc_score(y_val_l, y_prob_leak):.4f}  ← what we saw offline")
print(f"ROC-AUC in production:  {roc_auc_score(y_val_l, y_prob_production):.4f}  ← what actually happens")
print(f"Recall in validation:   0.54")
print(f"Recall in production:   {recall_score(y_val_l, y_pred_production):.2f}")
print()
print("This is model decay caused by leakage.")
print("The feature that boosted performance doesn't exist in real production.")

=== NAIVE vs LEAKY ===
Metric                    Naive      Leaky
------------------------------------------
Accuracy                 97.98%     98.12%
ROC-AUC                  0.9399     0.9556
Fraud Recall               0.49       0.54

Leaky model looks better on every metric.
Now let's simulate what happens in production...

=== PRODUCTION PERFORMANCE (30% new cards) ===
ROC-AUC in validation:  0.9556  ← what we saw offline
ROC-AUC in production:  0.8030  ← what actually happens
Recall in validation:   0.54
Recall in production:   0.38

This is model decay caused by leakage.
The feature that boosted performance doesn't exist in real production.


## Experiment 0B - Post Mortem

**The Leak:** card_fraud_rate computed on full dataset before train/test split.
**Offline AUC:** 0.9556 (looked great)
**Production AUC:** 0.8030 (collapsed on 30% new cards)
**Production Recall:** 0.38 (worse than the naive model)

**Why it's dangerous:** Leakage doesn't look wrong. It looks like improvement.
The model learns to cheat using information it won't have at inference time.

**How to detect it:**
- AUC suspiciously high after adding a feature? Investigate.
- Feature is derived from the target variable? Red flag.
- Always compute aggregates AFTER splitting, using only training data.

**Rule:** Any feature computed using the target variable across the full 
dataset before splitting = leakage. Always.

## Experiment 0C - The Wrong Metric
What we're doing: Showing that a model optimizing for accuracy on imbalanced data is equivalent to doing nothing. Then translating it into a business presentation a stakeholder would actually understand.

In [6]:
# ============================================
# EXPERIMENT 0C — THE WRONG METRIC
# ============================================

# The Dumbest Possible Model: predict nothing is fraud
# This is our "zero effort" baseline

y_dummy = np.zeros(len(y_val), dtype=int)  # predict everything as NOT fraud

print("=== THE DO-NOTHING MODEL ===")
print(f"Accuracy: {accuracy_score(y_val, y_dummy)*100:.2f}%")
print()
print(classification_report(y_val, y_dummy, target_names=['Not Fraud', 'Fraud']))

=== THE DO-NOTHING MODEL ===
Accuracy: 96.41%

              precision    recall  f1-score   support

   Not Fraud       0.96      1.00      0.98    113866
       Fraud       0.00      0.00      0.00      4242

    accuracy                           0.96    118108
   macro avg       0.48      0.50      0.49    118108
weighted avg       0.93      0.96      0.95    118108



C:\Users\Admin\anaconda3\envs\clean_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Admin\anaconda3\envs\clean_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\Admin\anaconda3\envs\clean_env\lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", 

In [7]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# ============================================
# FULL METRIC COMPARISON — ALL 3 MODELS
# ============================================

def business_cost(y_true, y_pred, fp_cost=500, fn_cost=50000):
    cm = confusion_matrix(y_true, y_pred)
    TN, FP, FN, TP = cm.ravel()
    return (FP * fp_cost) + (FN * fn_cost), FN, FP

# Costs
cost_dummy,  fn_d,  fp_d  = business_cost(y_val, y_dummy)
cost_naive,  fn_n,  fp_n  = business_cost(y_val, y_pred)
cost_leaky,  fn_l,  fp_l  = business_cost(y_val_l, y_pred_leak)

print("=== FULL MODEL COMPARISON ===")
print(f"{'Metric':<25} {'Do-Nothing':>12} {'Naive XGB':>12} {'Leaky XGB':>12}")
print("-" * 65)
print(f"{'Accuracy':<25} {'96.41%':>12} {'97.98%':>12} {'98.12%':>12}")
print(f"{'Fraud Recall':<25} {'0.00':>12} {'0.49':>12} {'0.54':>12}")
print(f"{'Fraud Precision':<25} {'0.00':>12} {'0.91':>12} {'0.90':>12}")
print(f"{'Missed Frauds':<25} {fn_d:>12,} {fn_n:>12,} {fn_l:>12,}")
print(f"{'False Alarms':<25} {fp_d:>12,} {fp_n:>12,} {fp_l:>12,}")
print(f"{'Business Cost (₹)':<25} {cost_dummy:>12,.0f} {cost_naive:>12,.0f} {cost_leaky:>12,.0f}")
print()

# The punchline
print("=== STAKEHOLDER SUMMARY ===")
print(f"Do-Nothing model:  Catches 0 frauds. Costs ₹{cost_dummy/1e7:.2f} crore.")
print(f"Naive XGB:         Catches {4242-fn_n} frauds. Costs ₹{cost_naive/1e7:.2f} crore.")
print(f"Leaky XGB:         Catches {4242-fn_l} frauds. Costs ₹{cost_leaky/1e7:.2f} crore.")
print()
print(f"Accuracy difference between Do-Nothing and Naive XGB: only {97.98-96.41:.2f}%")
print(f"Business cost difference: ₹{(cost_dummy-cost_naive)/1e7:.2f} crore")
print()
print("→ Accuracy hid a ₹10+ crore difference between doing nothing and doing something.")

=== FULL MODEL COMPARISON ===
Metric                      Do-Nothing    Naive XGB    Leaky XGB
-----------------------------------------------------------------
Accuracy                        96.41%       97.98%       98.12%
Fraud Recall                      0.00         0.49         0.54
Fraud Precision                   0.00         0.91         0.90
Missed Frauds                    4,242        2,176        1,955
False Alarms                         0          215          260
Business Cost (₹)          212,100,000  108,907,500   97,880,000

=== STAKEHOLDER SUMMARY ===
Do-Nothing model:  Catches 0 frauds. Costs ₹21.21 crore.
Naive XGB:         Catches 2066 frauds. Costs ₹10.89 crore.
Leaky XGB:         Catches 2287 frauds. Costs ₹9.79 crore.

Accuracy difference between Do-Nothing and Naive XGB: only 1.57%
Business cost difference: ₹10.32 crore

→ Accuracy hid a ₹10+ crore difference between doing nothing and doing something.


## Experiment 0C - Post Mortem

**The Wrong Metric:** Accuracy on imbalanced data.

- **Do-Nothing baseline:** 96.41% accuracy, ₹21.21 crore in losses.
- **Naive XGB:** 97.98% accuracy, ₹10.89 crore in losses.
- **Accuracy gap:** 1.57%. **Cost gap:** ₹10.32 crore.

**Rule:** On imbalanced data, always report:
- Recall (are we catching fraud?)
- Precision (are we blocking legitimate users unnecessarily?)
- Business Cost (what does each error actually cost?)
- PR-AUC (how does the model perform across all thresholds?)

Accuracy is for balanced datasets. This dataset is never balanced.